In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# Data load
print("\nData load ho raha hai...")
df = pd.read_csv("/content/drive/MyDrive/hadoop_openstack/hadoop_logs_parsed.csv")
print(f"Shape: {df.shape}")

# Features
le_app = LabelEncoder()
le_container = LabelEncoder()
df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])

features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features].values
y = df['label'].values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Normal only
X_train_normal = X_train[y_train == 0]

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Normal train: {len(X_train_normal):,}")
print(f"Test anomalies: {(y_test==1).sum():,}")
print("\nSab ready hai! ✅")

Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Data load ho raha hai...
Shape: (393433, 7)

X_train: (314746, 3)
X_test: (78687, 3)
Normal train: 20,228
Test anomalies: 73,630

Sab ready hai! ✅


In [2]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Conv1D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from imblearn.over_sampling import SMOTE
import numpy as np

results_dl = {}

# ============================================
# Model 1: LSTM Autoencoder
# ============================================
print("1. LSTM Autoencoder training...")

X_train_lstm = X_train.reshape((X_train.shape[0], 1, 3))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, 3))
X_train_normal_lstm = X_train_lstm[y_train == 0]

inputs = Input(shape=(1, 3))
encoded = LSTM(32, activation='relu', return_sequences=False)(inputs)
repeated = RepeatVector(1)(encoded)
decoded = LSTM(32, activation='relu', return_sequences=True)(repeated)
output = TimeDistributed(Dense(3))(decoded)

autoencoder = Model(inputs, output)
autoencoder.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True)

history = autoencoder.fit(
    X_train_normal_lstm, X_train_normal_lstm,
    epochs=20, batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop], verbose=1)

# Evaluate
X_test_pred = autoencoder.predict(X_test_lstm, batch_size=512, verbose=0)
mse = np.mean(np.power(X_test_lstm - X_test_pred, 2), axis=(1, 2))
threshold = np.percentile(mse[y_test==0], 95)
y_pred_lstm = (mse > threshold).astype(int)
roc_lstm = roc_auc_score(y_test, mse)

results_dl['LSTM Autoencoder'] = {
    'roc_auc': roc_lstm,
    'report': classification_report(y_test, y_pred_lstm,
                target_names=['Normal', 'Anomaly'])
}
print(f"LSTM ROC-AUC: {roc_lstm:.4f} ✅")

# ============================================
# Model 2: CNN Classifier
# ============================================
print("\n2. CNN Classifier training...")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

X_train_cnn = X_train_sm.reshape((X_train_sm.shape[0], 3, 1))
X_test_cnn = X_test.reshape((X_test.shape[0], 3, 1))

model_cnn = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu',
           input_shape=(3, 1)),
    Dropout(0.2),
    Conv1D(filters=64, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_cnn.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

early_stop2 = EarlyStopping(monitor='val_loss', patience=3,
                             restore_best_weights=True)

history_cnn = model_cnn.fit(
    X_train_cnn, y_train_sm,
    epochs=20, batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop2], verbose=1)

# Evaluate
y_prob_cnn = model_cnn.predict(X_test_cnn, batch_size=512, verbose=0).flatten()
y_pred_cnn = (y_prob_cnn > 0.5).astype(int)
roc_cnn = roc_auc_score(y_test, y_prob_cnn)

results_dl['CNN Classifier'] = {
    'roc_auc': roc_cnn,
    'report': classification_report(y_test, y_pred_cnn,
                target_names=['Normal', 'Anomaly'])
}
print(f"CNN ROC-AUC: {roc_cnn:.4f} ✅")

# ============================================
# Final Comparison
# ============================================
print("\n" + "=" * 45)
print("HADOOP — DEEP LEARNING RESULTS")
print("=" * 45)
for model, res in results_dl.items():
    print(f"\n{model} — ROC-AUC: {res['roc_auc']:.4f}")
    print(res['report'])

1. LSTM Autoencoder training...
Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 77ms/step - loss: 2.0069 - val_loss: 1.7792
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.6935 - val_loss: 1.2276
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7840 - val_loss: 0.3195
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1563 - val_loss: 0.0660
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0509 - val_loss: 0.0383
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0327 - val_loss: 0.0275
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0244 - val_loss: 0.0205
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0180 - val_loss: 0.0149
Epoch 9/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0129 - val_loss: 0.0104
Epoch 10/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0089 - val_loss: 0.0070
Epoch 11/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0061 - val_loss: 0.0047
Epoch 12/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1036/1036 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8077 - loss: 0.4316 - val_accuracy: 0.9565 - val_loss: 0.3243
Epoch 2/20
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8447 - loss: 0.3461 - val_accuracy: 0.9832 - val_loss: 0.2654
Epoch 3/20
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8668 - loss: 0.3028 - val_accuracy: 0.9832 - val_loss: 0.2377
Epoch 4/20
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8878 - loss: 0.2684 - val_accuracy: 0.9892 - val_loss: 0.1897
Epoch 5/20
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9051 - loss: 0.2390 - val_accuracy: 0.9871 - val_loss: 0.2242
Epoch 6/20
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9130 - loss: 0.2194 - val_accuracy: 0.9864 - val_loss: 0.1904
Epoch 7/20
1036/1036 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9175 - loss: 0.2060 - val_accuracy: 0.9452 - val_loss: 0.2802
CNN ROC-AUC: 0.9617 ✅

HADOOP — DEEP LEARNING RESULTS

LSTM Autoencoder — ROC-AUC: 0.6519
          

In [3]:
import json, os

save_dir = "/content/drive/MyDrive/hadoop_openstack"

hadoop_dl_results = {
    "CNN_Classifier":   {"ROC_AUC": 0.9617, "Recall": 0.84, "F1": 0.91},
    "LSTM_Autoencoder": {"ROC_AUC": 0.6519, "Recall": 0.29, "F1": 0.45}
}

with open(f"{save_dir}/hadoop_dl_results.json", "w") as f:
    json.dump(hadoop_dl_results, f, indent=4)

print("Saved! ✅")
print(json.dumps(hadoop_dl_results, indent=4))

Saved! ✅
{
    "CNN_Classifier": {
        "ROC_AUC": 0.9617,
        "Recall": 0.84,
        "F1": 0.91
    },
    "LSTM_Autoencoder": {
        "ROC_AUC": 0.6519,
        "Recall": 0.29,
        "F1": 0.45
    }
}
